# 05 — Proširenje #2: Kriva F1 vs veličina train skupa

Testiramo centralnu hipotezu tema-rada (Tashfeen): **attention modeli zahtevaju
više podataka da bi se isplatili.**

Treniramo M0 (baseline) i M1 (attention+ELU) na rastućim podskupovima
train skupa (25%, 50%, 75%, 100%), evaluiramo na ISTOM zamrznutom test setu,
i crtamo krivu F1 vs veličina.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src import config
from src.data_loading import make_dataset, _read_split
from src.models import build_model
from src.train import (
    get_device, train_model, find_best_threshold,
    get_subset_ids,
)

In [ ]:
device = get_device()
print(f"Device: {device}")

all_train_ids = _read_split("train")
fractions = config.DATA_FRACTIONS
print(f"Train ukupno: {len(all_train_ids)} slika")
print(f"Frakcije: {fractions}")
for f in fractions:
    sub = get_subset_ids(all_train_ids, f)
    print(f"  {f*100:.0f}%: {len(sub)} slika → {len(sub)*12} crop-ova")

In [ ]:
val_ds = make_dataset(split="val", form="full", augment=False)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

results = {"fraction": [], "model": [], "best_f1": [], "best_thresh": []}

## Trening po frakcijama

Za svaku frakciju treniramo M0 (baseline) i M1 (attention+ELU).

In [ ]:
all_metrics = {}

for frac in fractions:
    subset_ids = get_subset_ids(all_train_ids, frac)
    n_crops = len(subset_ids) * 12
    print(f"\n{'='*60}")
    print(f"Frakcija: {frac*100:.0f}% ({len(subset_ids)} slika, {n_crops} crop-ova)")
    print(f"{'='*60}")

    for model_name, use_attn in [("M0", False), ("M1", True)]:
        print(f"\n--- {model_name} ({frac*100:.0f}%) ---")

        train_ds = make_dataset(
            image_ids=subset_ids, form="crops", augment=True
        )
        bs = config.BATCH_SIZE_ATTENTION if use_attn else config.BATCH_SIZE_BASELINE
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=0)

        model = build_model(attention=use_attn, activation="elu")
        model = model.to(device)

        model, metrics = train_model(
            model, train_loader, val_loader,
            num_epochs=config.NUM_EPOCHS,
            lr=config.LEARNING_RATE,
            device=device,
        )

        best_thresh, thresh_res = find_best_threshold(model, val_loader, device=device)
        best_f1 = max(f1 for _, f1 in thresh_res)

        results["fraction"].append(frac)
        results["model"].append(model_name)
        results["best_f1"].append(best_f1)
        results["best_thresh"].append(best_thresh)

        key = f"{model_name}_{frac*100:.0f}pct"
        all_metrics[key] = metrics

        print(f"{model_name} @ {frac*100:.0f}%: F1={best_f1:.4f}, thresh={best_thresh:.2f}")

## Kriva F1 vs veličina train skupa

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for model_name, marker, color in [("M0", "o", "tab:blue"), ("M1", "s", "tab:orange")]:
    mask = [m == model_name for m in results["model"]]
    fracs = [results["fraction"][i] for i, m in enumerate(mask) if m]
    f1s   = [results["best_f1"][i]  for i, m in enumerate(mask) if m]
    sizes = [int(f * len(all_train_ids)) for f in fracs]

    ax.plot(sizes, f1s, marker=marker, color=color, linewidth=2,
            label=f"{model_name} ({'baseline' if model_name == 'M0' else 'attention+ELU'})")

ax.set_xlabel("Broj train slika", fontsize=12)
ax.set_ylabel("Piksel F1 (val)", fontsize=12)
ax.set_title("F1 vs veličina train skupa — M0 (baseline) vs M1 (attention)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Tabela rezultata

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df["n_images"] = (df["fraction"] * len(all_train_ids)).astype(int)
df["n_crops"] = df["n_images"] * 12
print(df[["model", "fraction", "n_images", "n_crops", "best_f1", "best_thresh"]].to_string(index=False))

## Zaključak

Na osnovu grafika iznad možemo da zaključimo da li attention model zahteva
više podataka da bi se isplatio u poređenju sa baseline-om — centralna
hipoteza tema-rada (Tashfeen, Santiago et al. 2024).

*(Popuniti nakon izvršavanja)*